In [5]:
from langchain_core.prompts import PromptTemplate
from langchain_community.chat_models.tongyi import ChatTongyi
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

model=ChatTongyi(model="qwen3-max")

prompt=PromptTemplate.from_template(
    "你需要根据会话历史回应问题对话历史{chat_history},用户提问：{input}, 请回答"
)

def print_prompt(p):
    print("="*10,p,"="*10)
    return p

str_parser=StrOutputParser()

base_chain=prompt |print_prompt | model | str_parser
store={}
def get_history(session_id):
    if session_id not in store:
        store[session_id]=InMemoryChatMessageHistory()
    return store[session_id]

conversation_chain =RunnableWithMessageHistory(
    base_chain,
    get_history,
input_messages_key="input",
history_messages_key="chat_history"
)

if __name__ == "__main__":
    session_config={
        "configurable":{
            "session_id":"user001"
        }
    }
    res=conversation_chain.invoke({
        "input":"peter有一个猫",
    }, session_config)

    print(res)
    res=conversation_chain.invoke({
        "input":"lisa有一个狗",
    }, session_config)

    print(res)
    res=conversation_chain.invoke({
        "input":"一共几个宠物",
    }, session_config)

    print(res)

========== text='你需要根据会话历史回应问题对话历史[],用户提问：peter有一个猫, 请回答' ==========
Peter有一只猫。
========== text="你需要根据会话历史回应问题对话历史[HumanMessage(content='peter有一个猫', additional_kwargs={}, response_metadata={}), AIMessage(content='Peter有一只猫。', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],用户提问：lisa有一个狗, 请回答" ==========
Lisa有一只狗。
========== text="你需要根据会话历史回应问题对话历史[HumanMessage(content='peter有一个猫', additional_kwargs={}, response_metadata={}), AIMessage(content='Peter有一只猫。', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='lisa有一个狗', additional_kwargs={}, response_metadata={}), AIMessage(content='Lisa有一只狗。', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],用户提问：一共几个宠物, 请回答" ==========
根据对话历史，Peter有一只猫，Lisa有一只狗，所以一共有2个宠物。
